# 28 - Saved dry scores at common conditional-FMR ceilings

**CPU Python Colab, Run All after26.** Only Drive and standard numpy/pandas are needed. No02/04, source images, torch/GPU, NBIS, training or27 dependency. Reads26's verified published completed-evaluation pointer and original aliased score units. No old artifacts are modified.

Primary results remain26's ORIGINAL FIXED INPUT THRESHOLDS. This supplementary descriptive ROC operating-point analysis uses the SAME saved scores at three predeclared common conditional-FMR ceilings: **0.1%, 0.5%, 1%**. For each DB/condition/method, choose the smallest nonnegative integer threshold with empirical FMR among MATCHABLE impostors <= target; score >= threshold accepts. Reuse the frozen NB08 calibration/rate helpers verified through26. Ties are not split; there is no interpolation, randomized tie acceptance or assumed continuous ROC.

These are common **ceilings**, NOT exactly equal achieved FMR or FAR. Report target, achieved FMR, impostor count, one-count resolution, allowed/accepted false-match counts, threshold, FNMR, attempt FAR/FRR, failure rates and matchable coverage. If the false-match budget is zero, flag it; if ties undershoot, flag that. No matchable impostors -> explicit unavailable row/warning, not invented threshold or original-input fallback. Missing genuine conditional metrics remain unavailable. Aggregate metrics require complete six-DB/three-dry-level coverage, not silent nan means.

Compare input, fixed mixed L1 seed20261110, ALL THREE dry checkpoints, per severity and equal-weight dry aggregate; identity separately. Include byte-distinct sensitivity at the same full-pair selected threshold; never recalibrate on the sensitivity view. Recompute fixed-threshold rates against26 before secondary analysis. No reranking/reselection of checkpoints, best seed or routing.

Thresholds are chosen and measured on the SAME repeatedly used development scores: optimistic descriptive operating points, NOT an independently calibrated evaluation or deployment thresholds. FMR is conditional on matchable impostors; attempt FAR and failure coverage can differ across methods. These score distributions do not prove identity-safe restoration or causal mechanisms. No significance/CI claims from this recalibration;26's fixed-threshold intervals do NOT apply here. No test/alternate scores opened.

Drive: `analyses/dry_common_fmr_v1/<hash>/`. Completion: **DRY COMMON FMR ANALYSIS COMPLETE**. Interpret achieved FMR, FNMR, attempt FRR/FAR and failures jointly; do not announce equal-FAR superiority based only on a common target.


In [ ]:
import ast
import hashlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
EVALUATION_ROOT = PROJECT_ROOT / 'evaluations' / 'dry_initialization_validation_v1'
TARGETS = (0.001, 0.005, 0.01)

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False)+'\n').encode()

def write_verified(path, data):
    pending = path.with_suffix(path.suffix+'.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failure: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False,lineterminator='\n').encode())

def verify_files(directory, hashes):
    for name, digest in hashes.items():
        if file_digest(directory/name) != digest:
            raise RuntimeError(f'Frozen artifact changed: {directory/name}')


In [ ]:
pointer = json.loads((EVALUATION_ROOT/'completed_evaluation.json').read_text(encoding='utf-8'))
EVAL_DIR = Path(pointer['directory'])
if (EVAL_DIR.parent != EVALUATION_ROOT
        or file_digest(EVAL_DIR/'configuration.json') != pointer['configuration_sha256']
        or file_digest(EVAL_DIR/'completion.json') != pointer['completion_sha256']):
    raise RuntimeError('Notebook26 published evaluation pointer/integrity mismatch.')
configuration = json.loads((EVAL_DIR/'configuration.json').read_text(encoding='utf-8'))
completion = json.loads((EVAL_DIR/'completion.json').read_text(encoding='utf-8'))
METHODS = ('input','l1_seed20261110','dry_l1_seed20261110','dry_l1_seed20261111','dry_l1_seed20261112')
CONDITIONS = ('identity/none','dry_like/mild','dry_like/moderate','dry_like/severe')
if (completion['status'] != 'dry_initialization_validation_complete'
        or completion['configuration_sha256'] != pointer['configuration_sha256']
        or completion['variants_available'] != 20 or tuple(completion['methods']) != METHODS
        or configuration['version'] != 'dry_initialization_validation_v1'
        or tuple('/'.join(condition) for condition in configuration['conditions']) != CONDITIONS):
    raise RuntimeError('Expected complete26 five-method/four-condition recognition.')
verify_files(EVAL_DIR, completion['file_sha256'])
helper_file = EVAL_DIR/'frozen_matching_helpers.py.txt'
if file_digest(helper_file) != configuration['frozen_helpers_sha256']:
    raise RuntimeError('Frozen recognition helper source changed.')
helper_source = helper_file.read_bytes().decode('utf-8')
helper_nodes = {node.name: node for node in ast.parse(helper_source).body if isinstance(node,ast.FunctionDef)}
functions_source = '\n\n'.join(ast.get_source_segment(helper_source,helper_nodes[name])
    for name in ('calibrate_threshold','metrics_at_threshold'))
helpers = dict(np=np,pd=pd,math=math)
exec(compile(functions_source,str(helper_file),'exec'),helpers)
calibrate_threshold, metrics_at_threshold = helpers['calibrate_threshold'],helpers['metrics_at_threshold']
units, source_rows = {}, []
for unit, provenance in completion['unit_provenance'].items():
    condition, method = provenance['condition'],provenance['method']
    family,level = condition.split('/')
    directory = Path(provenance['directory'])
    if (condition not in CONDITIONS or method not in METHODS or unit != f'{family}_{level}_{method}'
            or (condition,method) in units or provenance['source_variant'] != directory.name
            or file_digest(directory/'completion.json') != provenance['completion_sha256']):
        raise RuntimeError(f'Invalid26 unit provenance: {unit}')
    marker = json.loads((directory/'completion.json').read_text(encoding='utf-8'))
    if (marker['configuration_sha256'] != provenance['configuration_sha256']
            or marker['variant'] != directory.name
            or set(marker['file_sha256']) != {'scores.csv','extraction.csv','templates_and_logs.zip'}):
        raise RuntimeError(f'Unit completion differs: {unit}')
    verify_files(directory,marker['file_sha256'])
    frame = pd.read_csv(directory/'scores.csv')
    if (frame.empty or frame['pair_id'].duplicated().any() or frame['matchable'].dtype != bool
            or frame['duplicate_sensitivity_eligible'].dtype != bool or set(frame['label']) != {0,1}):
        raise RuntimeError(f'Invalid pair/label/boolean columns: {unit}')
    valid = frame.loc[frame['matchable'],'score']
    if (not np.isfinite(valid).all() or (valid < 0).any()
            or not np.equal(valid,np.floor(valid)).all() or frame.loc[~frame['matchable'],'score'].notna().any()):
        raise RuntimeError(f'Invalid saved scores/rejected attempt encoding: {unit}')
    frame = frame.sort_values('pair_id').reset_index(drop=True)
    units[(condition,method)] = frame
    source_rows.append({'unit':unit,'directory':str(directory),'scores_sha256':file_digest(directory/'scores.csv'),
        'completion_sha256':provenance['completion_sha256']})
if set(units) != {(condition,method) for condition in CONDITIONS for method in METHODS}:
    raise RuntimeError('Missing26 score units.')
canonical = units[('identity/none','input')]
columns = ['pair_id','dataset','db','label','probe_path','gallery_path','probe_identity','duplicate_sensitivity_eligible']
for key, frame in units.items():
    if len(frame) != completion['pairs_per_variant'] or not frame[columns].equals(canonical[columns]):
        raise RuntimeError(f'Pair metadata/coverage differs: {key}')
databases = sorted(set(zip(canonical['dataset'],canonical['db'])))
if len(databases) != 6:
    raise RuntimeError('Expected six source databases.')
fixed_thresholds = {(dataset,db): configuration['fixed_thresholds'][f'{dataset}_{db}'] for dataset,db in databases}
prior_metrics = pd.read_csv(EVAL_DIR/'validation_metrics.csv')
print('Verified20 saved score units; no images, torch, NBIS or training.')


In [ ]:
def ceiling_point(frame, target):
    if not math.isfinite(target) or not 0 <= target < 1:
        raise ValueError('Expected finite conditional-FMR target in [0,1).')
    genuine, impostor = frame['label'].eq(1),frame['label'].eq(0)
    if not genuine.any() or not impostor.any():
        raise RuntimeError('Both genuine and impostor attempts are required.')
    valid = frame.loc[impostor & frame['matchable'],'score'].to_numpy()
    base = {'target_conditional_fmr':target,'matchable_impostors':len(valid),
        'genuine_matchable_fraction':float(frame.loc[genuine,'matchable'].mean()),
        'impostor_matchable_fraction':float(frame.loc[impostor,'matchable'].mean())}
    if not len(valid):
        print('WARNING: no matchable impostors; conditional-FMR operating point unavailable.')
        return {**base,'status':'unavailable_no_matchable_impostors','threshold':None,
            'one_count_fmr_resolution':None,'allowed_false_matches':None,'accepted_false_matches':None,
            'attempt_frr':None,'attempt_far':None,'conditional_fnmr':None,'conditional_fmr':None}
    threshold = calibrate_threshold(valid,target)
    values = metrics_at_threshold(frame,threshold)
    allowed = math.floor(target*len(valid)+1e-12)
    accepted = int((valid >= threshold).sum())
    if accepted > allowed or values['conditional_fmr'] > target+1e-12:
        raise RuntimeError('Empirical conditional FMR ceiling exceeded.')
    if threshold > 0 and int((valid >= threshold-1).sum()) <= allowed:
        raise RuntimeError('Threshold is not the smallest integer satisfying the ceiling.')
    status = 'zero_false_match_budget' if allowed == 0 else (
        'ties_undershoot_false_match_budget' if accepted < allowed else 'budget_attained')
    if values['conditional_fnmr'] is None:
        print('WARNING: no matchable genuine pairs; conditional FNMR unavailable, all attempts retained.')
    return {**base,**values,'status':status,'one_count_fmr_resolution':1/len(valid),
        'allowed_false_matches':allowed,'accepted_false_matches':accepted,
        'conditional_genuine_status':'available' if values['conditional_fnmr'] is not None else 'unavailable_no_matchable_genuine'}


In [ ]:
fixed_rows, secondary_rows = [],[]
for condition in CONDITIONS:
    for method in METHODS:
        frame = units[(condition,method)]
        for (dataset,db),group in frame.groupby(['dataset','db'],sort=True):
            threshold = fixed_thresholds[(dataset,db)]
            values = metrics_at_threshold(group,threshold)
            previous = prior_metrics[(prior_metrics.condition == condition) & (prior_metrics.method == method)
                & (prior_metrics.dataset == dataset) & (prior_metrics.db == db)
                & (prior_metrics.threshold_policy == 'original_fixed') & (prior_metrics.view == 'all_impressions')]
            if len(previous) != 1:
                raise RuntimeError('Missing/duplicate26 primary metric.')
            for name,value in values.items():
                old = previous.iloc[0][name]
                if not ((value is None and pd.isna(old)) or (value is not None and np.isfinite(old) and abs(value-old)<=1e-10)):
                    raise RuntimeError(f'Primary fixed-threshold recomputation differs: {condition}/{method}/{dataset}/{db}/{name}')
            fixed_rows.append({'condition':condition,'method':method,'dataset':dataset,'db':db,**values})
            for target in TARGETS:
                full = ceiling_point(group,target)
                secondary_rows.append({'condition':condition,'method':method,'dataset':dataset,'db':db,
                    'view':'all_impressions',**full})
                sensitivity = group[group['duplicate_sensitivity_eligible']]
                if full['threshold'] is None:
                    sensitivity_values = {key:None for key in ('threshold','attempt_frr','attempt_far','conditional_fnmr','conditional_fmr')}
                else:
                    sensitivity_values = metrics_at_threshold(sensitivity,full['threshold'])
                secondary_rows.append({'condition':condition,'method':method,'dataset':dataset,'db':db,
                    'view':'byte_distinct_sensitivity_same_full_threshold','target_conditional_fmr':target,
                    'status':'not_recalibrated_same_full_threshold' if full['threshold'] is not None else full['status'],
                    **sensitivity_values})
secondary = pd.DataFrame(secondary_rows)
aggregate_rows = []
for (condition,method,target,view),group in secondary.groupby(
        ['condition','method','target_conditional_fmr','view'],sort=True):
    if len(group) != 6:
        raise RuntimeError('Incomplete database coverage in secondary analysis.')
    row = {'scope':condition,'method':method,'target_conditional_fmr':target,'view':view,'conditions':1}
    for metric in ('attempt_frr','attempt_far','conditional_fnmr','conditional_fmr'):
        count = int(group[metric].notna().sum())
        row[f'available_{metric}_databases'] = count
        row[metric] = float(group[metric].mean()) if count == 6 else None
        if count != 6:
            print(f'WARNING: incomplete six-DB {metric}: {condition}/{method}/{target}/{view}; no partial mean reported.')
    aggregate_rows.append(row)
condition_points = pd.DataFrame(aggregate_rows)
for (method,target,view),group in condition_points[condition_points.scope.str.startswith('dry_like/')].groupby(
        ['method','target_conditional_fmr','view'],sort=True):
    if len(group) != 3:
        raise RuntimeError('Incomplete dry condition coverage.')
    row = {'scope':'dry_like','method':method,'target_conditional_fmr':target,'view':view,'conditions':3}
    for metric in ('attempt_frr','attempt_far','conditional_fnmr','conditional_fmr'):
        complete = group[metric].notna().all()
        row[metric] = float(group[metric].mean()) if complete else None
        row[f'available_{metric}_databases'] = int(group[f'available_{metric}_databases'].min())
    aggregate_rows.append(row)
aggregate = pd.DataFrame(aggregate_rows)
comparison_rows = []
for (scope,target,view),group in aggregate.groupby(['scope','target_conditional_fmr','view'],sort=True):
    points = group.set_index('method')
    if set(points.index) != set(METHODS):
        raise RuntimeError('Secondary comparison missing a predeclared method.')
    for baseline in ('input','l1_seed20261110'):
        for method in METHODS[2:]:
            before,after = points.loc[baseline],points.loc[method]
            row = {'scope':scope,'target_conditional_fmr':target,'view':view,'baseline_method':baseline,'method':method}
            for metric in ('attempt_frr','attempt_far','conditional_fnmr','conditional_fmr'):
                left,right = before[metric],after[metric]
                row[f'delta_{metric}_pp'] = 100*float(right-left) if pd.notna(left) and pd.notna(right) else None
            comparison_rows.append(row)
comparisons = pd.DataFrame(comparison_rows)
print('Supplementary common-ceiling points, NOT equal achieved FMR/FAR:')
print(aggregate[(aggregate.scope.isin(['dry_like','identity/none'])) & aggregate.view.eq('all_impressions')].to_string(index=False))
print('Dry differences versus fixed mixed L1; empirical recalibration on same scores, no confidence claim:')
print(comparisons[comparisons.scope.eq('dry_like') & comparisons.view.eq('all_impressions')
    & comparisons.baseline_method.eq('l1_seed20261110')].to_string(index=False))


In [ ]:
analysis_config = {'version':'dry_common_fmr_v1','targets':TARGETS,
    'evaluation_configuration_sha256':pointer['configuration_sha256'],
    'evaluation_completion_sha256':pointer['completion_sha256'],
    'source_scores':source_rows,'frozen_helpers_sha256':configuration['frozen_helpers_sha256'],
    'ceiling_helper_sha256':hashlib.sha256(functions_source.encode()).hexdigest(),
    'methods':METHODS,'conditions':CONDITIONS,'numpy_version':np.__version__,'pandas_version':pd.__version__,
    'scope':'descriptive empirical FMR ceilings on same development scores, not exact equal FMR/FAR, not independent calibration'}
data = json_bytes(analysis_config)
sha = hashlib.sha256(data).hexdigest()
REPORT_DIR = PROJECT_ROOT/'analyses'/'dry_common_fmr_v1'/sha[:20]
REPORT_DIR.mkdir(parents=True,exist_ok=True)
path = REPORT_DIR/'configuration.json'
if path.exists() and path.read_bytes()!=data:
    raise RuntimeError('Frozen common-FMR analysis changed.')
write_verified(path,data)
write_csv(REPORT_DIR/'primary_fixed_threshold_recheck.csv',pd.DataFrame(fixed_rows))
write_csv(REPORT_DIR/'empirical_fmr_operating_points.csv',secondary)
write_csv(REPORT_DIR/'macro_common_fmr_points.csv',aggregate)
write_csv(REPORT_DIR/'common_fmr_comparisons.csv',comparisons)
write_csv(REPORT_DIR/'consumed_score_units.csv',pd.DataFrame(source_rows))
write_verified(REPORT_DIR/'frozen_rate_calibration_helpers.py.txt',functions_source.encode())
files = ['configuration.json','primary_fixed_threshold_recheck.csv','empirical_fmr_operating_points.csv',
    'macro_common_fmr_points.csv','common_fmr_comparisons.csv','consumed_score_units.csv','frozen_rate_calibration_helpers.py.txt']
write_verified(REPORT_DIR/'completion.json',json_bytes({'status':'dry_common_fmr_analysis_complete',
    'configuration_sha256':sha,'file_sha256':{name:file_digest(REPORT_DIR/name) for name in files},
    'training_performed':False,'primary_thresholds_changed':False,'inference_performed':False,'test_evaluated':False,
    'secondary_thresholds_fit_on_same_scores':True,'equal_achieved_fmr_claim':False}))
print('DRY COMMON FMR ANALYSIS COMPLETE:',REPORT_DIR)
print('Send all-impression common-ceiling points/differences and resolution/achieved-count diagnostics. Primary26 thresholds unchanged.')
